# Registro de experimentos de mi proyecto con MLflow

Una corrida por cada modelo de la Semana 10 (línea base, modelo lineal, ensamblaje y red neuronal), con la misma preparación y la misma división por fecha de `pipeline_proyecto.py`. Al final marco la corrida que elijo como final y compruebo que se puede cargar y reproduce su métrica.

In [1]:
import hashlib
import time

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.tracking import MlflowClient
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

from pipeline_proyecto import FECHA_CORTE, RUTA_DATOS, cargar_y_dividir, construir_preparacion

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()

NOMBRE_EXPERIMENTO = "ventas-walmart-m5"
mlflow.set_experiment(NOMBRE_EXPERIMENTO)


def huella(ruta):
    """SHA-256 del archivo de datos: si cambia una coma, cambia la huella."""
    sha = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            sha.update(bloque)
    return sha.hexdigest()


HUELLA_DATOS = huella(RUTA_DATOS)
print("Huella de los datos:", HUELLA_DATOS[:16], "...")

2026/11/16 17:14:20 INFO mlflow.tracking.fluent: Experiment with name 'ventas-walmart-m5' does not exist. Creating a new experiment.


Huella de los datos: 535d9ec679352ca7 ...


## La función que registra cada corrida

In [2]:
def registrar(nombre, modelo):
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    with mlflow.start_run(run_name=nombre) as corrida:
        # hiperparametros del modelo (sin los objetos anidados)
        params = {k: v for k, v in modelo.get_params().items() if isinstance(v, (int, float, str, bool, tuple, type(None)))}
        mlflow.log_params(params)

        # como se dividieron los datos y con que archivo
        mlflow.log_param("division", "por fecha")
        mlflow.log_param("fecha_corte", FECHA_CORTE)
        mlflow.log_param("semilla", 42)
        mlflow.log_param("huella_datos", HUELLA_DATOS)

        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio

        rmse_ent = root_mean_squared_error(y_ent, pipeline.predict(X_ent))
        rmse_prueba = root_mean_squared_error(y_prueba, pipeline.predict(X_prueba))
        mlflow.log_metric("rmse_entrenamiento", rmse_ent)
        mlflow.log_metric("rmse_prueba", rmse_prueba)
        mlflow.log_metric("segundos", segundos)

        # el pipeline completo, con la preparacion ajustada adentro
        mlflow.sklearn.log_model(pipeline, "modelo")
    print(f"{nombre:28s} prueba {rmse_prueba:.4f}   {segundos:6.2f} s   run_id {corrida.info.run_id}")
    return corrida.info.run_id

## Las cuatro corridas

In [3]:
ids = {}
ids["linea base"] = registrar("linea base", DummyRegressor(strategy="median"))
ids["ridge"] = registrar("ridge alpha=10", Ridge(alpha=10.0))
ids["bosque"] = registrar("bosque min_samples_leaf=20", RandomForestRegressor(
    n_estimators=100, min_samples_leaf=20, random_state=42, n_jobs=-1))
ids["red"] = registrar("red neuronal (32, 16)", MLPRegressor(
    hidden_layer_sizes=(32, 16), early_stopping=True, max_iter=200, random_state=42))

2026/11/16 17:14:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/11/16 17:14:28 INFO mlflow.tracking._tracking_service.client: 🏃 View run linea base at: http://127.0.0.1:5050/#/experiments/2/runs/1a3780da729b433487c7543930ce3425.
2026/11/16 17:14:28 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5050/#/experiments/2.


linea base                   prueba 3.8348     0.23 s   run_id 1a3780da729b433487c7543930ce3425


2026/11/16 17:14:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/11/16 17:14:33 INFO mlflow.tracking._tracking_service.client: 🏃 View run ridge alpha=10 at: http://127.0.0.1:5050/#/experiments/2/runs/b00d28cc8aad4af9a3cbd099d50f02b3.
2026/11/16 17:14:33 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5050/#/experiments/2.


ridge alpha=10               prueba 3.0188     0.26 s   run_id b00d28cc8aad4af9a3cbd099d50f02b3


2026/11/16 17:15:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/11/16 17:15:33 INFO mlflow.tracking._tracking_service.client: 🏃 View run bosque min_samples_leaf=20 at: http://127.0.0.1:5050/#/experiments/2/runs/f1dfecd918604501a046660b925adde5.
2026/11/16 17:15:33 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5050/#/experiments/2.


bosque min_samples_leaf=20   prueba 2.9332    45.76 s   run_id f1dfecd918604501a046660b925adde5


2026/11/16 17:17:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/11/16 17:17:02 INFO mlflow.tracking._tracking_service.client: 🏃 View run red neuronal (32, 16) at: http://127.0.0.1:5050/#/experiments/2/runs/fee18dd30a054429bcd1c2c077d05e65.
2026/11/16 17:17:02 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5050/#/experiments/2.


red neuronal (32, 16)        prueba 2.9572    68.81 s   run_id fee18dd30a054429bcd1c2c077d05e65


## La corrida final

In [4]:
corridas = mlflow.search_runs(experiment_names=[NOMBRE_EXPERIMENTO])

# la de menor error en prueba
final = corridas.sort_values("metrics.rmse_prueba").iloc[0]
RUN_FINAL = final["run_id"]
print("Corrida final:", final["tags.mlflow.runName"], "| run_id:", RUN_FINAL)

MlflowClient().set_tag(RUN_FINAL, "candidato", "final")

Corrida final: bosque min_samples_leaf=20 | run_id: f1dfecd918604501a046660b925adde5


## La tabla de corridas

In [5]:
# se exporta despues de marcar la final, para que la etiqueta quede en la tabla
corridas = mlflow.search_runs(experiment_names=[NOMBRE_EXPERIMENTO])
corridas.to_csv("corridas_proyecto.csv", index=False)
corridas[["run_id", "tags.mlflow.runName", "tags.candidato", "metrics.rmse_entrenamiento",
          "metrics.rmse_prueba", "metrics.segundos"]].sort_values("metrics.rmse_prueba")

,run_id,tags.mlflow.runName,tags.candidato,metrics.rmse_entrenamiento,metrics.rmse_prueba,metrics.segundos
1,f1dfecd918604501a046660b925adde5,bosque min_samples_leaf=20,final,2.823285,2.933216,45.755410
0,fee18dd30a054429bcd1c2c077d05e65,"red neuronal (32, 16)",None,2.818621,2.957225,68.807321
2,b00d28cc8aad4af9a3cbd099d50f02b3,ridge alpha=10,None,3.087925,3.018843,0.262568
3,1a3780da729b433487c7543930ce3425,linea base,None,3.874910,3.834765,0.228862


In [6]:
# cargo el modelo desde el registro y compruebo que da la metrica que quedo guardada
modelo_final = mlflow.sklearn.load_model(f"runs:/{RUN_FINAL}/modelo")
rmse_recalculado = root_mean_squared_error(y_prueba, modelo_final.predict(X_prueba))
rmse_registrado = final["metrics.rmse_prueba"]
print("RMSE de prueba registrado:", round(rmse_registrado, 6))
print("RMSE de prueba recalculado:", round(rmse_recalculado, 6))
assert np.isclose(rmse_recalculado, rmse_registrado), "el modelo cargado no reproduce la metrica"
print("Reproduce la metrica registrada")

RMSE de prueba registrado: 2.933216
RMSE de prueba recalculado: 2.933216
Reproduce la metrica registrada
